# Indexing TREC Robust 2004 by OpenSearch for Sparse Encoder Model

Chunks and sparse-encodes each passage entirely server-side via a remote
sparse-encoding model registered in OpenSearch.

- [disks45/nocr/trec-robust-2004](https://ir-datasets.com/disks45.html#disks45/nocr/trec-robust-2004)
- Prerequisite: [ml_model_registration.ipynb](ml_model_registration.ipynb)

In [1]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv beautifulsoup4

In [2]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [3]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'qIJ28Ej2TCC6_LI88zZLlw',
 'name': '4db878c40bab',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2026-02-07T07:54:31.169913465Z',
             'build_hash': 'bbc94f0bdc3a759011e6529ecfe52840856f91a3',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.2',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.5.0'}}


### Index a Corpus for SPLADE Model

Encoding runs on the remote model host, so no local GPU / sentence-transformers is needed here.

In [4]:
import ir_datasets
dataset_name = "disks45/nocr/trec-robust-2004"
dataset = ir_datasets.load(dataset_name)

In [5]:
index_name = "trec_robust_2004_splade"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

### Sparse Encoding of Chunks (server-side ingest pipeline)

Chunk **and** sparse-encode entirely inside OpenSearch. Bulk sends *raw* documents; a single ingest pipeline runs two processors in order:

1. `text_chunking` — splits `text` into passages in `text_chunks`.
2. `sparse_encoding` — calls the remote model on each passage and writes per-chunk `rank_features` to the nested `text_chunks_embedding` field.

In [ ]:
pipeline_id = "trec_robust_2004_chunk_sparse"
model_id = "your-model-id"  # sparse PASSAGE model (/embed/passages) from ml_model_registration

In [ ]:
def create_chunk_sparse_pipeline(
    pipeline_id: str,
    model_id: str,
    source_field: str = "text",
    chunk_field: str = "text_chunks",
    embedding_field: str = "text_chunks_embedding",
    token_limit: int = 384,
    overlap_rate: float = 0.2,
    tokenizer: str = "standard",
    batch_size: int = 16,
) -> dict:
    """
    Create (or update) an ingest pipeline that chunks then sparse-encodes text,
    fully server-side.

    Stage 1 (`text_chunking`) splits `source_field` into passages in `chunk_field`.
    Stage 2 (`sparse_encoding`) embeds each passage with the remote `model_id`,
    writing a nested list of rank_features to `embedding_field`.

    `batch_size` bundles that many *documents'* chunks into a single model call
    (batch ingestion), so the GPU processes them as one padded batch instead of
    one at a time. Actual texts per call ~= batch_size x avg chunks/doc.
    """
    body = {
        "description": "Chunk documents, then sparse-encode each passage",
        "processors": [
            {
                "text_chunking": {
                    "algorithm": {
                        "fixed_token_length": {
                            "token_limit": token_limit,
                            "overlap_rate": overlap_rate,
                            "tokenizer": tokenizer,
                        }
                    },
                    "field_map": {source_field: chunk_field},
                }
            },
            {
                "sparse_encoding": {
                    "model_id": model_id,
                    "field_map": {chunk_field: embedding_field},
                    "batch_size": batch_size,
                }
            },
        ],
    }
    return client.ingest.put_pipeline(id=pipeline_id, body=body)

response = create_chunk_sparse_pipeline(pipeline_id, model_id, batch_size=64)
pprint.pprint(response)

Bulk indexing

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
        "text_chunks": { "type": "text" },
        # Per-chunk sparse vectors produced by the sparse_encoding processor.
        # Chunking yields a list of passages, so the embeddings must be `nested`.
        "text_chunks_embedding": {
            "type": "nested",
            "properties": {
                "sparse_encoding": { "type": "rank_features" }
            }
        },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

In [ ]:
# Cap doc length so a single very long doc can't explode into hundreds of chunks
# (each chunk is sparse-encoded on the model host; unbounded chunks/doc OOMs it).
# ~40k chars ~= 6-8k standard tokens ~= 25 chunks, safely under the default
# analyze.max_token_count (10k), so no doc is rejected either.
MAX_DOC_CHARS = 10_000

def prepare_documents(dataset):
    """
    Yield raw bulk actions. Chunking + encoding happen server-side in the
    ingest pipeline, so we only ship docid/title/text.

    ir_datasets already parses each document into clean `title` / `body`
    fields. The corpus mixes FBIS, FR94, FT and LATIMES markup (e.g. FBIS
    uses <TI>/<TEXT>, not <HEADLINE>/<P>), so we use those parsed fields
    directly instead of re-parsing `marked_up_doc` ourselves — the custom
    BeautifulSoup parser silently produced empty text for FBIS/FR94 docs,
    which then had nothing to encode.
    """
    for doc in dataset.docs_iter():
        text = doc.body.replace("\n", " ")[:MAX_DOC_CHARS]
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "title": doc.title,
                "text": text
            }
        }

In [ ]:
from opensearchpy.helpers import streaming_bulk

# Batch inference is configured server-side in the pipeline's `sparse_encoding`
# processor (batch_size=64). OpenSearch 3.x removed the `_bulk?batch_size=` query
# param (2.x only), so we must NOT pass one here — it 400s the whole request.

# Total for the progress bar (docs_count is instant; fall back to a full id scan).
try:
    total = dataset.docs_count()
except Exception:
    total = sum(1 for _ in dataset.docs_iter())

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in streaming_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        pipeline=pipeline_id,
        chunk_size=128,                # 2 x processor batch_size; bar advances every 128 docs
        request_timeout=300,
        max_retries=3,
        initial_backoff=2,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed doc
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)}")
if errors:
    pprint.pprint(errors[:3])          # inspect the first few errors

---
### (Optional) Re-index documents missing from the first pass

If a run was interrupted, diff the corpus against what's actually in the index
and re-index just the missing ids. The retry reuses the same truncation
(`MAX_DOC_CHARS`) as the main pass, so no `analyze.max_token_count` change is
needed — and it won't overload the encoder with a runaway chunk count.

In [7]:
from opensearchpy.helpers import scan

# All ids actually in the index (_source disabled -> fast)
indexed = set()
for hit in scan(
    client,
    index=index_name,
    query={"query": {"match_all": {}}, "_source": False},
    size=5000,
):
    indexed.add(hit["_id"])

# All ids the dataset should have produced
all_ids = {doc.doc_id for doc in dataset.docs_iter()}

missing = sorted(all_ids - indexed)
print(f"indexed: {len(indexed)},  missing: {len(missing)}")
print(missing[:10])

indexed: 527935,  missing: 220
['FBIS3-13309', 'FBIS3-14832', 'FBIS3-1509', 'FBIS3-1658', 'FBIS3-1771', 'FBIS3-1790', 'FBIS3-20796', 'FBIS3-20797', 'FBIS3-20801', 'FBIS3-22119']


In [8]:
# Re-index the docs identified as missing by the scan diff, printing every error.
from opensearchpy.helpers import streaming_bulk

# Same truncation as the main pass (MAX_DOC_CHARS) — no cap raising, so a single
# huge doc can't blow up the encoder's per-call batch.
MAX_DOC_CHARS = 10_000
def prepare_missing(dataset, missing_ids):
    docstore = dataset.docs_store()
    for doc_id in missing_ids:
        doc = docstore.get(doc_id)
        text = doc.body.replace("\n", " ")[:MAX_DOC_CHARS]
        yield {
            "_id": doc_id,
            "_source": {"docid": doc_id, "title": doc.title, "text": text},
        }

retry_ok, retry_failed = 0, []
with tqdm(total=len(missing), desc="Re-indexing") as bar:
    for ok, item in streaming_bulk(
        client,
        prepare_missing(dataset, missing),
        index=index_name,
        pipeline=pipeline_id,
        chunk_size=128,
        request_timeout=300,
        raise_on_error=False,
        raise_on_exception=False,
    ):
        bar.update(1)              # advances per actually-processed doc
        retry_ok += ok
        if not ok:
            retry_failed.append(item)

print(f"retried ok: {retry_ok}, still failing: {len(retry_failed)}\n")

for item in retry_failed:          # full error detail for every failure
    pprint.pprint(item)
    print("-" * 80)

Re-indexing: 100%|██████████| 220/220 [00:16<00:00, 13.47it/s]

retried ok: 220, still failing: 0

